# InfoSleuth SniffTest — GRPO Training (Sanity Check)

Fine-tune **Qwen3-1.7B** on the SniffTest misinformation investigation environment
using **GRPO** (Group Relative Policy Optimization) via TRL.

**Purpose:** Verify that the training reward curve goes up across all three difficulty levels.
**Time:** ~45–60 min on A100
**GPU:** A100 required (Colab Pro)
Based on the [TRL OpenEnv Wordle example](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb).

---

### Four reward signals
| Signal | Range | What it captures |
|---|---|---|
| `accuracy` | 0 / 1 | Correct verdict label |
| `evidence` | 0–1 | Fraction of key evidence sources opened |
| `efficiency` | 0–1 | Fewer steps = higher score |
| `format` | 0–1 | Valid JSON output on every turn |

The total reward per episode is the **sum** of all four (max 4.0).

In [ ]:
# Requires A100 GPU — Runtime → Change runtime type → A100
!pip install -Uq "trl>=0.17.0" "openenv-core>=0.2.2" transformers datasets \
    accelerate "vllm>=0.4.0" pydantic matplotlib openai
print("✅ Dependencies installed")


## Step 1 — Get the Code into Colab

1. **Push** your project to GitHub (private or public).
2. **Set `GITHUB_REPO_URL`** in the next cell.
3. Run the cell.

For private repos use a token URL:
`https://<TOKEN>@github.com/YOUR_USER/YOUR_REPO.git`
Generate tokens at: https://github.com/settings/tokens

**Required repo layout:**
```
<repo>/
  snifftest_env/
    data/claims_dataset.json   ← required
    server/snifftest_environment.py
    server/adversarial.py
    models.py
    inference.py
    ...
```

In [ ]:
import os, sys, subprocess

GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # ← SET THIS

result = subprocess.run(
    ["git", "clone", "--depth=1", GITHUB_REPO_URL, "repo"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    if "already exists" not in result.stderr:
        raise RuntimeError(f"Clone failed:\n{result.stderr}")
    subprocess.run(["git", "-C", "repo", "pull", "-q"])
    print("Repo already present — pulled latest.")
else:
    print("✅ Repo cloned.")

SNIFFTEST_PATH = os.path.abspath("repo/snifftest_env")
assert os.path.exists(SNIFFTEST_PATH), (
    f"Path not found: {SNIFFTEST_PATH}\n"
    "Check that your repo has a snifftest_env/ directory at the root."
)
if SNIFFTEST_PATH not in sys.path:
    sys.path.insert(0, SNIFFTEST_PATH)
print(f"✅ SNIFFTEST_PATH = {SNIFFTEST_PATH}")


In [ ]:
import json, textwrap, torch
from collections import defaultdict
from pathlib import Path

from server.snifftest_environment import SniffTestEnvironment
from models import InvestigateAction, SniffTestObservation

print("✅ Environment imports OK")

dataset_path = Path(SNIFFTEST_PATH) / "data" / "claims_dataset.json"
with open(dataset_path) as f:
    raw_scenarios = json.load(f)

counts = defaultdict(int)
for s in raw_scenarios:
    counts[s.get("difficulty", "unknown")] += 1

print(f"\n📊 Dataset: {len(raw_scenarios)} scenarios")
print(f"   easy={counts['easy']}  medium={counts['medium']}  hard={counts['hard']}")


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
gpu = torch.cuda.get_device_properties(0)
total_gb = round(gpu.total_memory / 1024 ** 3, 1)
start_memory = round(torch.cuda.max_memory_reserved() / 1024 ** 3, 3)
print(f"GPU:  {gpu.name}")
print(f"VRAM: {total_gb} GB total  |  {start_memory} GB already reserved")
if total_gb < 38:
    print("⚠️  A100 (40 GB) strongly recommended — current GPU may OOM during training.")
    print("   Go to: Runtime → Change runtime type → GPU → A100")
else:
    print("✅ GPU is A100 — good to go!")


## Step 2 — Environment Sanity Check

Three quick manual episodes (one per difficulty) to confirm the env is healthy before
committing to a full training run. Each episode takes a fixed 3-action script so there
is no model inference involved — pure env logic only.

In [ ]:
print("=" * 65)
print("  ENVIRONMENT SANITY CHECK  —  3 MANUAL EPISODES")
print("=" * 65)

sanity_env = SniffTestEnvironment()

for diff in ["easy", "medium", "hard"]:
    obs = sanity_env.reset(task_level=diff)
    print(f"\n🔍 [{diff.upper()}]")
    print(f"   Claim:   {obs.claim[:80]}...")
    print(f"   Sources: {len(obs.available_sources)} visible  |  "
          f"Steps remaining: {obs.steps_remaining}")

    # Minimal 3-step script: search → open first source → submit verdict (naive guess)
    src_id = obs.available_sources[0].source_id if obs.available_sources else None
    actions = [
        InvestigateAction(action_type="search", query="evidence fact-check"),
    ]
    if src_id:
        actions.append(
            InvestigateAction(action_type="open_source", source_id=src_id)
        )
    actions.append(
        InvestigateAction(
            action_type="submit_verdict",
            verdict="false",        # naive guess — we just want rewards to flow
            confidence=0.4,
            justification="Sanity check only — not a real investigation.",
        )
    )

    for act in actions:
        obs = sanity_env.step(act)
        if obs.done:
            break

    g = sanity_env.state.grade_result or {}
    print(f"   Reward:   {obs.reward:.4f}")
    print(f"   Accuracy: {g.get('accuracy', 0):.2f}  |  "
          f"Evidence: {g.get('evidence_alignment', 0):.2f}  |  "
          f"Efficiency: {g.get('efficiency', 0):.2f}  |  "
          f"Final: {g.get('final_score', 0):.4f}")
    print(f"   Truth label: {g.get('truth_label', '?')}  →  "
          f"Agent said: {g.get('agent_verdict', '?')}")

print("\n✅ Environment is healthy and grader is returning scores!")


## Step 3 — Load Model and Tokenizer


In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model:      {model_name}")
print(f"Vocab size: {tokenizer.vocab_size:,}")


## Step 4 — System Prompt and Observation Helpers

The system prompt is loaded directly from `inference.py` so training and evaluation
use the exact same prompt. `obs_to_text` converts a `SniffTestObservation` into a
user-turn message, and `parse_action` extracts a JSON action from model output.

In [ ]:
import importlib.util, os as _os

_spec = importlib.util.spec_from_file_location(
    "snifftest_inference",
    _os.path.join(SNIFFTEST_PATH, "inference.py"),
)
_mod = importlib.util.module_from_spec(_spec)
try:
    _spec.loader.exec_module(_mod)
    SYSTEM_PROMPT = _mod.SYSTEM_PROMPT
    print("✅ System prompt loaded from inference.py")
except Exception as _e:
    # Fallback — mirrors the inference.py prompt without importing it
    SYSTEM_PROMPT = textwrap.dedent('''
        You are an expert fact-checker investigating a claim. Use available tools methodically.

        Available actions (respond with a single JSON object on one line):
        {"action_type": "search", "query": "<search terms>"}
        {"action_type": "open_source", "source_id": "<src_id>"}
        {"action_type": "cross_reference", "source_ids": ["<id1>", "<id2>"]}
        {"action_type": "trace_origin", "source_id": "<src_id>"}
        {"action_type": "check_metadata", "source_id": "<src_id>"}
        {"action_type": "submit_verdict", "verdict": "true|false|misleading|unverifiable",
         "confidence": 0.9, "justification": "reasoning citing source IDs and domains"}

        Strategy: search -> open key sources -> cross-reference suspicious ones -> submit verdict.
        Respond with ONLY the JSON action -- no prose, no markdown fences.
    ''').strip()
    print(f"⚠️  Fallback system prompt used ({_e})")

print(f"Prompt: {len(SYSTEM_PROMPT)} chars")

MAX_TURNS = 10


def obs_to_text(obs, step: int) -> str:
    """Render a SniffTestObservation as a user-turn message for the model."""
    lines = [
        f"Step {step}/{MAX_TURNS}  |  Steps remaining: {obs.steps_remaining}",
        "",
        f"CLAIM: {obs.claim}",
        "",
        f"AVAILABLE SOURCES ({len(obs.available_sources)} discovered):",
    ]
    for s in obs.available_sources:
        tag = " [READ]" if s.retrieved else ""
        lines.append(f"  [{s.source_id}] {s.title} ({s.domain}){tag}")
        lines.append(f"    {s.snippet}")
    if obs.opened_content:
        lines += ["", "LAST OPENED:", obs.opened_content[:500]]
    if obs.cross_reference_result:
        lines += ["", "CROSS-REFERENCE:", obs.cross_reference_result[:300]]
    if obs.trace_result:
        lines += ["", "TRACE RESULT:", obs.trace_result[:300]]
    if obs.metadata_result:
        lines += ["", "METADATA CHECK:", obs.metadata_result[:300]]
    if obs.action_history:
        lines += ["", "ACTION HISTORY (last 5):"]
        for h in obs.action_history[-5:]:
            lines.append(f"  Step {h.step} [{h.action_type}]: {h.result_summary}")
    if obs.message:
        lines += ["", f"FEEDBACK: {obs.message}"]
    lines += ["", "Next action (JSON only):"]
    return "\n".join(lines)


def parse_action(text: str) -> tuple:
    """Extract a JSON action from model output. Returns (dict, is_valid_bool)."""
    text = text.strip()
    if text.startswith("```"):
        parts = text.split("```")
        text = parts[1] if len(parts) > 1 else text
        if text.startswith("json"):
            text = text[4:]
    text = text.strip()
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("{"):
            try:
                return json.loads(line), True
            except json.JSONDecodeError:
                pass
    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        return {"action_type": "search", "query": "claim evidence"}, False


print("obs_to_text and parse_action defined.")


## Step 5 — Rollout and Reward Functions

`rollout_once` plays one full SniffTest episode using the model under training.
It concatenates the token IDs from every turn into a single sequence — the same
pattern used in the Wordle reference notebook.

`rollout_func` is the entry point called by `GRPOTrainer`. Each prompt string is
the difficulty level (`"easy"`, `"medium"`, or `"hard"`).

In [ ]:
from trl.experimental.openenv import generate_rollout_completions

# A single shared environment — the WeaknessTracker accumulates across
# all training episodes so it can detect patterns.
train_env = SniffTestEnvironment()


def rollout_once(trainer, env, task_level: str) -> dict:
    """
    Play one full SniffTest episode with the current model.

    Returns a dict with:
      prompt_ids / completion_ids / logprobs  — flat token sequences (all turns concat'd)
      accuracy_reward   — 1.0 if verdict label is correct, else 0.0
      evidence_reward   — fraction of key evidence sources opened (0-1)
      efficiency_reward — grader efficiency score (1.0 at 3 steps, 0.0 at 10)
      format_reward     — fraction of turns with parseable JSON output (0-1)
    """
    obs = env.reset(task_level=task_level)

    all_prompt_ids: list = []
    all_completion_ids: list = []
    all_logprobs: list = []
    valid_actions = 0
    total_actions = 0

    for turn in range(MAX_TURNS):
        if getattr(obs, "done", False):
            break

        user_msg = obs_to_text(obs, turn + 1)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        out = generate_rollout_completions(trainer, [prompt_text])[0]
        all_prompt_ids.extend(out["prompt_ids"])
        all_completion_ids.extend(out["completion_ids"])
        all_logprobs.extend(out["logprobs"])

        completion_text = out.get("text") or tokenizer.decode(
            out["completion_ids"], skip_special_tokens=True
        )

        action_dict, valid = parse_action(completion_text)
        total_actions += 1
        valid_actions += int(valid)

        try:
            action = InvestigateAction(**action_dict)
        except Exception:
            action = InvestigateAction(action_type="search", query="claim evidence")
            valid_actions = max(0, valid_actions - 1)

        obs = env.step(action)

    grade = env.state.grade_result or {}
    fmt   = valid_actions / max(total_actions, 1)

    return {
        "prompt_ids":        all_prompt_ids,
        "completion_ids":    all_completion_ids,
        "logprobs":          all_logprobs,
        "accuracy_reward":   float(grade.get("accuracy",           0.0)),
        "evidence_reward":   float(grade.get("evidence_alignment", 0.0)),
        "efficiency_reward": float(grade.get("efficiency",         0.0)),
        "format_reward":     float(fmt),
    }


def rollout_func(prompts, trainer=None):
    """
    Entry point called by GRPOTrainer.
    Each element of `prompts` is a difficulty string from the training dataset.
    """
    batch = {k: [] for k in [
        "prompt_ids", "completion_ids", "logprobs",
        "accuracy_reward", "evidence_reward", "efficiency_reward", "format_reward",
    ]}
    for task_level in prompts:
        level = task_level.strip()
        if level not in ("easy", "medium", "hard"):
            level = "easy"
        ep = rollout_once(trainer=trainer, env=train_env, task_level=level)
        for key in batch:
            batch[key].append(ep[key])
    return batch


print("rollout_once and rollout_func defined.")
print(f"Shared train_env initialised (WeaknessTracker starts fresh).")


In [ ]:
# ── Four reward functions passed to GRPOTrainer ───────────────────────────
# GRPOTrainer calls each one and sums the outputs into the total reward.

def reward_accuracy(completions, **kw):
    """1.0 if the verdict label is correct, 0.0 otherwise. Primary signal."""
    return [float(r) for r in kw.get("accuracy_reward",   [0.0] * len(completions))]

def reward_evidence(completions, **kw):
    """Fraction of key evidence sources the agent opened before submitting verdict."""
    return [float(r) for r in kw.get("evidence_reward",   [0.0] * len(completions))]

def reward_efficiency(completions, **kw):
    """Grader efficiency: 1.0 if done in 3 steps, decays linearly to 0.0 at 10."""
    return [float(r) for r in kw.get("efficiency_reward", [0.0] * len(completions))]

def reward_format(completions, **kw):
    """Fraction of turns where the model produced parseable JSON (0-1)."""
    return [float(r) for r in kw.get("format_reward",     [0.0] * len(completions))]

print("Reward functions defined: accuracy, evidence, efficiency, format")
print("Max total reward per episode: 4.0  (all four signals at their maximum)")


## Step 6 — Dataset, Config, and Trainer


In [ ]:
from datasets import Dataset

# Each prompt is a difficulty string.  rollout_func passes it to env.reset(task_level=...).
# Equal representation across all three difficulty levels.
N_PER_LEVEL = 133   # 399 total → ~50 gradient steps (accum=8, batch=1)

TRAIN_DATASET = Dataset.from_dict({
    "prompt": (
        ["easy"]   * N_PER_LEVEL
        + ["medium"] * N_PER_LEVEL
        + ["hard"]   * N_PER_LEVEL
    )
})
print(f"Training dataset: {len(TRAIN_DATASET)} prompts  "
      f"({N_PER_LEVEL} easy / {N_PER_LEVEL} medium / {N_PER_LEVEL} hard)")


In [ ]:
from trl import GRPOConfig

OUTPUT_DIR = "snifftest-grpo-Qwen3-1.7B"

# ══ SANITY CHECK CONFIG (default) ══════════════════════════════════════════
# ~50 gradient steps · ~45-60 min on A100
# Goal: verify reward curve trends upward — not maximum performance.
grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=8,    # ~50 grad steps total
    per_device_train_batch_size=1,
    warmup_steps=5,
    num_generations=2,                # 2 rollouts per prompt → relative reward comparison
    max_completion_length=120,        # JSON actions are short (30-80 tokens)
    max_prompt_length=1400,
    use_vllm=True,
    vllm_mode="colocate",             # generation + training share the same A100
    vllm_gpu_memory_utilization=0.35,
    output_dir=OUTPUT_DIR,
    report_to="none",                 # plots are generated below; swap to "trackio" for live dash
    logging_steps=1,
    save_steps=25,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    push_to_hub=False,                # set True for full competition run
)

# ══ FULL COMPETITION RUN (uncomment for finals — ~2-3 h on A100) ══════════
# grpo_config = GRPOConfig(
#     num_train_epochs=3,
#     learning_rate=5e-6,
#     gradient_accumulation_steps=16,
#     per_device_train_batch_size=1,
#     warmup_steps=10,
#     num_generations=4,
#     max_completion_length=120,
#     max_prompt_length=1400,
#     use_vllm=True,
#     vllm_mode="colocate",
#     vllm_gpu_memory_utilization=0.5,
#     output_dir=OUTPUT_DIR,
#     report_to="trackio",
#     trackio_space_id=OUTPUT_DIR,
#     logging_steps=1,
#     save_steps=25,
#     gradient_checkpointing=True,
#     gradient_checkpointing_kwargs={"use_reentrant": False},
#     push_to_hub=True,
# )

n_grad = len(TRAIN_DATASET) // (
    grpo_config.gradient_accumulation_steps * grpo_config.per_device_train_batch_size
)
print(f"Output dir:       {OUTPUT_DIR}")
print(f"Gradient steps:   ~{n_grad}")
print(f"Est. time A100:   ~{n_grad * 1}–{int(n_grad * 1.5)} min")


In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[reward_accuracy, reward_evidence, reward_efficiency, reward_format],
    train_dataset=TRAIN_DATASET,
    args=grpo_config,
    rollout_func=rollout_func,
)
print("✅ GRPOTrainer created  (vLLM colocate mode)")


## Step 7 — Pre-training Baseline Evaluation

Evaluate the **untrained base model** on all three difficulties.
This establishes the before/after baseline for the training curve.

In [ ]:
import numpy as np


def evaluate_all_difficulties(trainer_obj, n_episodes: int = 5, label: str = "") -> dict:
    """
    Run n_episodes per difficulty level using the current model.
    Returns a dict: {difficulty: {accuracy, evidence, efficiency, final_score}}.
    """
    eval_env = SniffTestEnvironment()
    results  = {}
    divider  = "─" * 58

    print(f"\n{divider}")
    if label:
        print(f"  {label}")
    print(f"  {'DIFFICULTY':10}  {'ACCURACY':9}  {'EVIDENCE':9}  {'EFFICIENCY':11}  {'FINAL':7}")
    print(divider)

    for diff in ["easy", "medium", "hard"]:
        accs, evids, effs, finals = [], [], [], []
        for _ in range(n_episodes):
            rollout_once(trainer=trainer_obj, env=eval_env, task_level=diff)
            g = eval_env.state.grade_result or {}
            accs.append(g.get("accuracy",           0.0))
            evids.append(g.get("evidence_alignment", 0.0))
            effs.append(g.get("efficiency",          0.0))
            finals.append(g.get("final_score",       0.0))

        results[diff] = {
            "accuracy":    float(np.mean(accs)),
            "evidence":    float(np.mean(evids)),
            "efficiency":  float(np.mean(effs)),
            "final_score": float(np.mean(finals)),
        }
        r = results[diff]
        print(f"  {diff.upper():10}  {r['accuracy']:.3f}      {r['evidence']:.3f}      "
              f"{r['efficiency']:.3f}        {r['final_score']:.3f}")

    print(divider)
    return results


N_EVAL_EPISODES = 5
print(f"Running pre-training baseline  ({N_EVAL_EPISODES} episodes per difficulty)…")
pre_results = evaluate_all_difficulties(
    trainer, n_episodes=N_EVAL_EPISODES, label="PRE-TRAINING  (base Qwen3-1.7B)"
)


## Step 8 — Train!

Sanity-check config: ~45–60 min on A100.
The reward curve is plotted in Step 9 immediately after training finishes.

In [ ]:
gpu_props  = torch.cuda.get_device_properties(0)
mem_before = round(torch.cuda.max_memory_reserved() / 1024 ** 3, 2)
print(f"GPU memory before training: {mem_before} GB reserved / "
      f"{round(gpu_props.total_memory / 1024**3, 1)} GB total")
print()

trainer_stats = trainer.train()


In [ ]:
used_mem  = round(torch.cuda.max_memory_reserved() / 1024 ** 3, 2)
total_mem = round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1)
runtime   = round(trainer_stats.metrics["train_runtime"] / 60, 1)
print(f"Training time : {runtime} min")
print(f"Peak memory   : {used_mem} GB  ({round(used_mem / total_mem * 100, 1)}% of {total_mem} GB)")


## Step 9 — Training Curves


In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history

steps, total_r, r_acc, r_ev, r_eff, r_fmt = [], [], [], [], [], []
for entry in history:
    if "reward" not in entry:
        continue
    steps.append(entry.get("step", len(steps) + 1))
    total_r.append(entry.get("reward", 0))
    r_acc.append(entry.get("rewards/reward_accuracy",   0))
    r_ev.append( entry.get("rewards/reward_evidence",   0))
    r_eff.append(entry.get("rewards/reward_efficiency", 0))
    r_fmt.append(entry.get("rewards/reward_format",     0))

if not steps:
    print("⚠️  No reward entries in log_history — verify logging_steps=1 is set.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("SniffTest — GRPO Training Curves", fontsize=13, fontweight="bold")

    # Left panel: total reward
    axes[0].plot(steps, total_r, color="#1565C0", linewidth=2)
    axes[0].set_title("Mean Total Reward per Step  (sum of 4 signals, max 4.0)")
    axes[0].set_xlabel("Gradient step")
    axes[0].set_ylabel("Total reward")
    axes[0].set_ylim(bottom=0)
    axes[0].grid(True, alpha=0.3)

    # Right panel: per-signal breakdown
    axes[1].plot(steps, r_acc, color="#F44336", linewidth=1.5, label="accuracy")
    axes[1].plot(steps, r_ev,  color="#4CAF50", linewidth=1.5, label="evidence")
    axes[1].plot(steps, r_eff, color="#FF9800", linewidth=1.5, label="efficiency")
    axes[1].plot(steps, r_fmt, color="#9C27B0", linewidth=1.5, label="format")
    axes[1].set_title("Per-Signal Reward Breakdown")
    axes[1].set_xlabel("Gradient step")
    axes[1].set_ylabel("Reward (0–1 each)")
    axes[1].set_ylim(0, 1.05)
    axes[1].legend(loc="lower right")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → training_curve.png   ({len(steps)} data points)")


## Step 10 — Post-training Evaluation and Comparison


In [ ]:
print(f"Running post-training eval  ({N_EVAL_EPISODES} episodes per difficulty)…")
post_results = evaluate_all_difficulties(
    trainer, n_episodes=N_EVAL_EPISODES, label="POST-TRAINING  (fine-tuned Qwen3-1.7B)"
)

# Print delta summary
print()
print("  IMPROVEMENT SUMMARY")
print(f"  {'DIFFICULTY':10}  {'Δ ACCURACY':11}  {'Δ EVIDENCE':11}  {'Δ FINAL':10}")
print("  " + "─" * 48)
for diff in ["easy", "medium", "hard"]:
    da = post_results[diff]["accuracy"]    - pre_results[diff]["accuracy"]
    de = post_results[diff]["evidence"]    - pre_results[diff]["evidence"]
    df = post_results[diff]["final_score"] - pre_results[diff]["final_score"]
    arrow = lambda v: "↑" if v > 0.01 else ("↓" if v < -0.01 else "→")
    print(f"  {diff.upper():10}  {da:+.3f}  {arrow(da)}      "
          f"{de:+.3f}  {arrow(de)}      {df:+.3f}  {arrow(df)}")


In [ ]:
import numpy as np

diffs       = ["Easy", "Medium", "Hard"]
pre_final   = [pre_results[d.lower()]["final_score"] for d in diffs]
post_final  = [post_results[d.lower()]["final_score"] for d in diffs]
pre_acc     = [pre_results[d.lower()]["accuracy"]    for d in diffs]
post_acc    = [post_results[d.lower()]["accuracy"]   for d in diffs]

x = np.arange(len(diffs))
w = 0.32

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Before vs After GRPO Fine-tuning", fontsize=12, fontweight="bold")

for ax, (pre, post, title) in zip(axes, [
    (pre_final, post_final, "Final Score  (weighted, 0–1)"),
    (pre_acc,   post_acc,   "Accuracy  (correct verdict %)"),
]):
    b1 = ax.bar(x - w / 2, pre,  w, label="Before (base)",  color="#BBDEFB", edgecolor="#fff")
    b2 = ax.bar(x + w / 2, post, w, label="After (trained)", color="#1565C0", edgecolor="#fff")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(diffs)
    ax.set_ylim(0, 1.15)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    for bar in b1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=8, color="#555")
    for bar in b2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=8,
                fontweight="bold", color="#1565C0")

plt.tight_layout()
plt.savefig("before_after_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → before_after_comparison.png")


## Step 11 — Weakness Tracker Report

The `WeaknessTracker` has been silently logging every episode during training.
**Copy this output and share it** to generate targeted adversarial datasets for
the next training run.

For the actual competition event, this step is automated via the OpenAI API.
Until then, share the weakness summary here with Claude to get a new
`claims_dataset_adv.json`.

In [ ]:
wt      = train_env._weakness_tracker
history = wt.history
n       = len(history)

print("=" * 65)
print("  WEAKNESS TRACKER REPORT")
print("  Copy everything below this line and share with Claude")
print("  to generate adversarial datasets targeting these weaknesses.")
print("=" * 65)
print(f"  Episodes logged  : {n}")
print(f"  Min for analysis : {wt.MIN_EPISODES}")
print()

if n == 0:
    print("  No episodes recorded — check rollout_func is calling env correctly.")
elif n < wt.MIN_EPISODES:
    print(f"  Only {n} episodes — need {wt.MIN_EPISODES} to confirm weaknesses.")
    print("  Run more training steps or lower MIN_EPISODES for a quicker check.")
else:
    weaknesses = wt.get_weaknesses()
    print("  ── CONFIRMED WEAKNESSES ──────────────────────────────")
    if weaknesses:
        print(wt.summary_for_prompt())
    else:
        print("  None — model is above threshold on all tracked dimensions.")

    # Raw stats regardless
    avg_acc   = sum(ep.accuracy           for ep in history) / n
    avg_ev    = sum(ep.evidence_alignment for ep in history) / n
    t_rate    = sum(1 for ep in history if ep.timed_out)     / n
    by_diff   = {"easy": [], "medium": [], "hard": []}
    for ep in history:
        by_diff.get(ep.difficulty, []).append(ep.accuracy)

    print()
    print("  ── RAW EPISODE STATS ─────────────────────────────────")
    print(f"  Avg accuracy        : {avg_acc:.3f}")
    print(f"  Avg evidence align  : {avg_ev:.3f}")
    print(f"  Timeout rate        : {t_rate:.1%}")
    print()
    print("  ── PER-DIFFICULTY ACCURACY ───────────────────────────")
    for d, accs in by_diff.items():
        if accs:
            print(f"  {d.upper():8}  avg={sum(accs)/len(accs):.3f}  ({len(accs)} episodes)")

print()
print("=" * 65)
print("  HOW TO GENERATE ADVERSARIAL SCENARIOS (manual, pre-event):")
print("  1. Copy the weakness summary above.")
print("  2. Paste it to Claude with the message:")
print('     "Generate 10 adversarial SniffTest scenarios targeting: <paste>"')
print("  3. Save the JSON output to  data/claims_dataset_adv.json")
print("  4. Run the injection cell below, then rerun cells 18–30.")
print("=" * 65)


### Adversarial Dataset Injection  *(manual trigger)*

Run this cell **after** you have a generated adversarial dataset saved at the path below.
It injects the new scenarios into the environment's scenario pool and rebuilds the
training dataset with extra hard-difficulty prompts.  Then re-run cells 18 through 30
to do another training round.

In [ ]:
# ── SET THIS to your generated adversarial dataset path ──────────────────
ADV_PATH = "repo/snifftest_env/data/claims_dataset_adv.json"

if not os.path.exists(ADV_PATH):
    print(f"⚠️  File not found: {ADV_PATH}")
    print("   Generate adversarial scenarios first, then re-run this cell.")
else:
    with open(ADV_PATH) as _f:
        adv_scenarios = json.load(_f)

    # Inject into the live env's scenario pool
    train_env._all_scenarios.extend(adv_scenarios)
    print(f"✅ Injected {len(adv_scenarios)} adversarial scenarios.")
    print(f"   Total in pool: {len(train_env._all_scenarios)}")

    # Rebuild dataset with extra hard prompts to give adversarial scenarios more coverage
    N_HARD_EXTRA = len(adv_scenarios) * 2
    TRAIN_DATASET = Dataset.from_dict({
        "prompt": (
            ["easy"]   * N_PER_LEVEL
            + ["medium"] * N_PER_LEVEL
            + ["hard"]   * (N_PER_LEVEL + N_HARD_EXTRA)
        )
    })
    print(f"   Updated dataset: {len(TRAIN_DATASET)} prompts "
          f"(added {N_HARD_EXTRA} extra hard prompts)")
    print()
    print("   Next: re-run cells 20 (trainer) through 30 (comparison chart).")


## Step 12 — Save the Model


In [ ]:
trainer.save_model(OUTPUT_DIR)
print(f"✅ Model saved to: {OUTPUT_DIR}/")

# Uncomment for the full competition run once you have a HF repo:
# trainer.push_to_hub()
# print("✅ Model pushed to HuggingFace Hub.")

print()
print("Generated artefacts:")
for fname in ["training_curve.png", "before_after_comparison.png"]:
    if os.path.exists(fname):
        size_kb = os.path.getsize(fname) / 1024
        print(f"  {fname}  ({size_kb:.1f} KB)")
